# Cultist Wars — fixed Kaggle notebook

Attach the original cultist-nn-41710886.zip dataset. Select GPU T4 x2. Run all three cells in order. Setup handles both intact archives and Kaggle-expanded nested gzip files; existing setup is reusable. Return the final marked report and results ZIP. No arena submission.


In [ ]:
from pathlib import Path
import gzip, hashlib, json, shutil, subprocess, sys, zipfile

work = Path('/kaggle/working/cultist-nn-41710886')
inputs = Path('/kaggle/input')
expected = '8de82a9a36ea84159f137fb82ac27a820b469ffe63b5745c95293054e5a5e62c'

# Reuse an existing setup, or locate the attached dataset.
if not (work / 'manifest.json').is_file():
    candidates = [p.parent for p in inputs.rglob('manifest.json')
                  if p.is_file() and (p.parent / 'train.py').is_file()]
    if candidates:
        shutil.copytree(candidates[0], work, dirs_exist_ok=True)
    else:
        archives = [p for p in inputs.rglob('cultist-nn-41710886.zip') if p.is_file()]
        assert archives, 'Attach cultist-nn-41710886.zip as notebook input first.'
        with zipfile.ZipFile(archives[0]) as z:
            z.extractall('/kaggle/working')

# Kaggle may expand gzip into games.jsonl/games-2026-11-13.jsonl.
# Find the actual file by its verified uncompressed contents.
target = work / 'games.jsonl.gz'
payload = None
candidates = ([target] if target.is_file() else [])
for base in (work, inputs):
    candidates.extend(p for p in base.rglob('*')
                      if p.is_file() and 'games' in str(p.relative_to(base)).lower())
for path in candidates:
    data = path.read_bytes()
    if data.startswith(b'\x1f\x8b'):
        data = gzip.decompress(data)
    if hashlib.sha256(data).hexdigest() == expected:
        payload = data
        print('Verified replay:', path)
        break
assert payload is not None, 'Replay payload missing or changed. Report dataset filenames.'
if not target.is_file():
    target.write_bytes(gzip.compress(payload, mtime=0))

manifest = json.loads((work / 'manifest.json').read_text())
for name, digest in manifest.items():
    data = (work / name).read_bytes()
    if name == 'games.jsonl.gz':
        assert hashlib.sha256(gzip.decompress(data)).hexdigest() == expected, name
    else:
        assert hashlib.sha256(data).hexdigest() == digest, name
print('Bundle integrity: PASS', flush=True)
subprocess.run([sys.executable, '-u', str(work / 'prepare.py')], check=True)


In [ ]:
subprocess.run([
    sys.executable, str(work / 'train.py'),
    '--device', 'cuda:0', '--minutes', '12', '--epochs', '80'
], check=True)


In [ ]:
from IPython.display import FileLink, display
result = shutil.make_archive('/kaggle/working/cultist-nn-results', 'zip', work / 'output')
print('=== PASTE BACK START ===')
print((work / 'output/report.json').read_text())
print('=== PASTE BACK END ===')
display(FileLink(result))
